In [2]:
import requests
from bs4 import BeautifulSoup
import json
import re


# --------------------------------------------------
# Fetch page
# --------------------------------------------------
def call_API(url: str) -> str:
    payload = {}
    headers = {
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
    'accept-language': 'en-US,en;q=0.9',
    'cache-control': 'max-age=0',
    'dnt': '1',
    'if-none-match': 'W/"a8eca-uiKTMxbYuM9ZYzfZXYEH0YTk+18"',
    'priority': 'u=0, i',
    'referer': 'https://www.99acres.com/search/property/buy/ahmedabad-west?city=49&preference=S&area_unit=1&res_com=R',
    'sec-ch-ua': '"Not(A:Brand";v="8", "Chromium";v="144", "Google Chrome";v="144"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-user': '?1',
    'upgrade-insecure-requests': '1',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36',
    'Cookie': '99_ab=29; GOOGLE_SEARCH_ID=4084631769856564402; xAB=SuperControlGroup%3D17%3AN%2CtopMatchHandlingAB%3D66%3AD%2CBFdataremoval%3D29%3AY%2CseamlessLogin%3D56%3AY%2CEMAILOPTIONAL%3D48%3AY%2CMLSEARCHSRP%3D1%3AY%2CDSSimilarProperties%3D87%3AY%2CshowInhousePlayer%3D38%3AD%2CIATABSVF%3D36%3AD%2CVSRAlgoDemandShaping%3D88%3AY%2CBUILDERFLOORSRP%3D99%3AN%2CMLSEARCHMONET%3D78%3AY%2CNEARBYSRP%3D31%3AY%2CppfTemplatePostingV2%3D48%3AY%2CbrokerSupplyRef%3D40%3AY%2CownerEmailOptional%3D3%3AY%2CppfCommSoftPosting%3D51%3AN; session_source=DIRECT; landmark_toast=true; _gcl_au=1.1.349653589.1769856570; _gid=GA1.2.649331308.1769856571; _fbp=fb.1.1769856570831.879743020712544429; _clck=1yly54u%5E2%5Eg36%5E0%5E2222; showCookieBanner=1; _hjSessionUser_3171461=eyJpZCI6IjlmNzI5MTZkLTUwMTMtNTVjNC1iMmQzLTZhMTkxOGYzNGFhYiIsImNyZWF0ZWQiOjE3Njk4NTY1NzEwMTksImV4aXN0aW5nIjp0cnVlfQ==; 99_ab=29; acceptedMobileDsiclaimer=true; hp_bcf_data=; _hjSession_3171461=eyJpZCI6ImZkZTVkNGE2LTlhMDUtNDFhZS1hNmJlLTExNzkzYTdjMDljZCIsImMiOjE3Njk4NjMzOTkyMjYsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowfQ==; session30m=eyJ0b2tlbklkIjoiY2FiNWViMjItYmYxYi00ZjY0LWJiZmUtNWUyYWNmNGI2OGUzIiwiaXNzdWVEYXRlIjoxNzY5ODY1NTkzMTA3fQ; sessionno=4; _sess_id=s5BzqCncDAFyNb1ouAe8Jpa6%2FKIHd5Zxc1kYfiVH4NHODNSp2oppo4AlOiGLI35Z1oOs2A5blHu%2FR83G1PY3EQ%3D%3D; _gat_UA-224016-1=1; _uetsid=85d4df90fe9211f0b03a6d756b7dcf35; _uetvid=85d55230fe9211f08d2da945ba1deab4; _ga=GA1.1.477790739.1769856571; _ga_9QHC0XEKPS=GS2.1.s1769863361$o3$g1$t1769865608$j60$l0$h0; _clsk=1yhhs3a%5E1769865608136%5E13%5E0%5Ev.clarity.ms%2Fcollect; 99_ab=74; GOOGLE_SEARCH_ID=2104631769791185608; sessionno=3; xAB=SuperControlGroup%3D17%3AN%2CtopMatchHandlingAB%3D66%3AD%2CBFdataremoval%3D29%3AY%2CseamlessLogin%3D56%3AY%2CEMAILOPTIONAL%3D48%3AY%2CMLSEARCHSRP%3D1%3AY%2CDSSimilarProperties%3D87%3AY%2CshowInhousePlayer%3D38%3AD%2CIATABSVF%3D36%3AD%2CVSRAlgoDemandShaping%3D88%3AY%2CBUILDERFLOORSRP%3D99%3AN%2CMLSEARCHMONET%3D78%3AY%2CNEARBYSRP%3D31%3AY%2CppfTemplatePostingV2%3D48%3AY%2CbrokerSupplyRef%3D40%3AY%2CownerEmailOptional%3D3%3AY%2CppfCommSoftPosting%3D51%3AN'
    }

    response = requests.request("GET", url, headers=headers, data=payload, timeout=30)
    response.raise_for_status()
    print(f"✅ Fetched page successfully: {response.status_code}")
    return response.text


# --------------------------------------------------
# Extract __initialData__ (price lives here)
# --------------------------------------------------
def extract_initial_data(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")
    pattern = re.compile(r"window\.__initialData__\s*=\s*(\{.*?\})\s*;", re.DOTALL)

    for script in soup.find_all("script"):
        if script.string:
            match = pattern.search(script.string)
            if match:
                print("✅ window.__initialData__ extracted")
                return json.loads(match.group(1))

    return {}


# --------------------------------------------------
# Extract JSON-LD (property + location lives here)
# --------------------------------------------------
def extract_json_ld(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")

    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
            if data.get("@type") in [
                "SingleFamilyResidence",
                "Residence",
                "Apartment"
            ]:
                print("✅ Property JSON-LD extracted")
                return data
        except Exception:
            pass

    return {}


# --------------------------------------------------
# Parse property from JSON-LD
# --------------------------------------------------
def parse_property_from_jsonld(jld: dict) -> dict:
    area_val = None
    area_unit = None

    floor = jld.get("floorSize")
    if isinstance(floor, str):
        parts = floor.split()
        if len(parts) == 2:
            area_val, area_unit = parts

    return {
        "property_type": jld.get("@type"),
        "bhk": jld.get("numberOfRooms"),
        "builtup_area": area_val,
        "area_unit": area_unit,
        "image": jld.get("image"),
    }


# --------------------------------------------------
# Parse location from JSON-LD
# --------------------------------------------------
def parse_location_from_jsonld(jld: dict) -> dict:
    address = jld.get("address", {})
    geo = jld.get("geo", {})

    return {
        "locality": address.get("streetAddress"),
        "city": address.get("addressLocality"),
        "state": address.get("addressRegion"),
        "country": address.get("addressCountry"),
        "latitude": geo.get("latitude"),
        "longitude": geo.get("longitude"),
    }


# --------------------------------------------------
# Parse pricing from __initialData__
# --------------------------------------------------
def parse_pricing(initial_data: dict) -> dict:
    price = (
        initial_data
        .get("pageData", {})
        .get("price", {})
    )

    return {
        "min_price": price.get("min"),
        "max_price": price.get("max"),
        "price_label": price.get("valueLabel"),
        "authentic": price.get("authentic"),
    }


# --------------------------------------------------
# Final payload
# --------------------------------------------------
def build_payload(html: str) -> dict:
    initial_data = extract_initial_data(html)
    json_ld = extract_json_ld(html)

    return {
        "property": parse_property_from_jsonld(json_ld),
        "pricing": parse_pricing(initial_data),
        "location": parse_location_from_jsonld(json_ld),
        "nearby_places_of_interest": {},  # NOT available via requests
    }


# --------------------------------------------------
# MAIN
# --------------------------------------------------
def main():
    url = "https://www.99acres.com/4-bhk-bedroom-independent-house-villa-for-sale-in-heritage-villa-76-kasindra-ahmedabad-south-3798-sq-ft-npspid-D85437812"

    html = call_API(url)

    payload = build_payload(html)

    print("\n🎯 FINAL EXTRACTED DATA\n")
    print(json.dumps(payload, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


✅ Fetched page successfully: 200
✅ window.__initialData__ extracted
✅ Property JSON-LD extracted

🎯 FINAL EXTRACTED DATA

{
  "property": {
    "property_type": "SingleFamilyResidence",
    "bhk": "4",
    "builtup_area": "3798",
    "area_unit": "sqft",
    "image": "https://imagecdn.99acres.com/media1/32596/11/651931969O-1766983879062.jpg"
  },
  "pricing": {
    "min_price": null,
    "max_price": null,
    "price_label": null,
    "authentic": null
  },
  "location": {
    "locality": "Kasindra",
    "city": "Ahmedabad South",
    "state": null,
    "country": "India",
    "latitude": "22.902978",
    "longitude": "72.46739"
  },
  "nearby_places_of_interest": {}
}


In [8]:
import requests
from bs4 import BeautifulSoup
import json
import re
from typing import Dict, Any, Optional, List

# --------------------------------------------------
# 1️⃣ Scraper / API Call
# --------------------------------------------------
def call_API(url: str) -> str:
    """
    Fetches the HTML content of the 99acres property page.
    """
    headers = {
        'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
        'accept-language': 'en-US,en;q=0.9',
        'cache-control': 'max-age=0',
        'dnt': '1',
        'priority': 'u=0, i',
        'referer': 'https://www.99acres.com/',
        'sec-ch-ua': '"Chromium";v="124", "Google Chrome";v="124", "Not-A.Brand";v="99"',
        'sec-ch-ua-mobile': '?0',
        'sec-ch-ua-platform': '"Windows"',
        'sec-fetch-dest': 'document',
        'sec-fetch-mode': 'navigate',
        'sec-fetch-site': 'same-origin',
        'sec-fetch-user': '?1',
        'upgrade-insecure-requests': '1',
        'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        print(f"✅ Fetched page successfully: {response.status_code}")
        return response.text
    except Exception as e:
        print(f"❌ Error fetching page: {e}")
        return ""

# --------------------------------------------------
# 2️⃣ Extract window.__initialData__ (Fixed Logic)
# --------------------------------------------------
def extract_initial_data(html: str) -> Dict[str, Any]:
    """
    Extracts the JSON blob from window.__initialData__ using string parsing
    to handle nested JSON correctly.
    """
    soup = BeautifulSoup(html, "lxml")

    # Iterate over scripts to find the data variable
    for script in soup.find_all("script"):
        if script.string and "window.__initialData__" in script.string:
            content = script.string.strip()
            
            # Locate the start of the JSON object
            marker = "window.__initialData__="
            start_index = content.find(marker)
            if start_index == -1:
                continue
            
            # Move index to the start of the actual JSON '{'
            json_start = content.find("{", start_index)
            if json_start == -1:
                continue
            
            # The JSON typically ends at the end of the script string.
            # We treat the rest of the string as the candidate JSON.
            # Sometimes it ends with a semicolon ';', so we strip that.
            json_str = content[json_start:]
            if json_str.endswith(";"):
                json_str = json_str[:-1]
                
            try:
                data = json.loads(json_str)
                print("✅ window.__initialData__ extracted successfully")
                return data
            except json.JSONDecodeError:
                # If naive extraction fails, try a slightly more aggressive trim
                # or continue to next script
                continue

    raise RuntimeError("❌ window.__initialData__ not found in page HTML")

# --------------------------------------------------
# 3️⃣ Dynamically locate property node
# --------------------------------------------------
def find_property_node(data: Any) -> Optional[Dict[str, Any]]:
    """
    Recursively find the dictionary that represents property details.
    Looks for keys specific to 99acres property objects (Prop_Id, Price, etc).
    """
    if isinstance(data, dict):
        keys = data.keys()
        # "Prop_Id" and "Price" are strong indicators of the main property node
        if "Prop_Id" in keys and "Price" in keys:
            return data
        
        # Recursive search in values
        for key, value in data.items():
            found = find_property_node(value)
            if found:
                return found
                
    elif isinstance(data, list):
        for item in data:
            found = find_property_node(item)
            if found:
                return found

    return None

# --------------------------------------------------
# 4️⃣ Parsers (Updated Keys)
# --------------------------------------------------
def parse_property_details(node: dict) -> dict:
    return {
        "listing_id": node.get("Prop_Id"),
        "title": node.get("Start_Text") or node.get("propertyTitle"),
        "property_type": node.get("Property_Text") or node.get("Property_Type"),
        "project_name": node.get("Building_Name"),
        "bhk": node.get("Bedroom_Num") or node.get("bedrooms"),
        "bathrooms": node.get("Bathroom_Num") or node.get("bathrooms"),
        # Area often appears as Price_Per_Unit_Area_Text, but explicit built-up area might be in 'Super_Area'
        # or require calculation/lookup in other nodes.
        "area_text": node.get("Super_Area") or node.get("builtUpArea"), 
        "furnishing": node.get("Furnish_Label") or node.get("furnishing"),
        "possession": node.get("Availability_Text") or node.get("availabilityStatus"),
        "ownership": node.get("Ownership_Label"),
        "is_verified": node.get("isVerified"),
    }

def parse_pricing(node: dict) -> dict:
    return {
        "price": node.get("Price"),
        "price_per_sqft": node.get("Price_Per_Unit_Area_Text"),
        "total_price_text": node.get("Price_Text"), 
        "booking_amount": node.get("Booking_Amount"),
        "is_negotiable": node.get("isNegotiable")
    }

def parse_location(node: dict) -> dict:
    return {
        "city_id": node.get("City"),
        "locality_id": node.get("Locality_Id") or node.get("localityid"),
        "address_label": node.get("headerDescriptionAddressInfo"),
        "latitude": node.get("Latitude"),
        "longitude": node.get("Longitude"),
    }

def parse_images(node: dict) -> list:
    # 99acres uses 'PROPERTY_IMAGES' which is a list of image URLs
    images = node.get("PROPERTY_IMAGES") or node.get("images") or []
    
    # Ensure we return a clean list of strings
    if images and isinstance(images[0], dict):
        return [img.get('url', '') for img in images]
    return images

def parse_nearby(node: dict) -> list:
    # "nearByPlacesOfInterest" is often directly in the property node
    places = node.get("nearByPlacesOfInterest", [])
    results = []
    for p in places:
        if isinstance(p, dict):
            results.append({
                "name": p.get("text"),
                "category": p.get("category"),
                "distance": p.get("distance")
            })
    return results

# --------------------------------------------------
# 5️⃣ Build final payload
# --------------------------------------------------
def build_payload(initial_data: dict) -> dict:
    property_node = find_property_node(initial_data)

    if not property_node:
        # Fallback search strategies if recursive search fails
        try:
            # Common path in some 99acres responses
            property_node = initial_data["pageData"]["custominfo"]["payload"]["property"]
        except KeyError:
            raise RuntimeError("❌ Property data node not found in extracted JSON")

    print(f"✅ Property node detected (ID: {property_node.get('Prop_Id')})")

    return {
        "property": parse_property_details(property_node),
        "pricing": parse_pricing(property_node),
        "location": parse_location(property_node),
        "images": parse_images(property_node),
        "nearby_places": parse_nearby(property_node),
        # Useful for debugging if specific keys are missing
        "raw_sample": {k: property_node[k] for k in list(property_node.keys())[:5]} 
    }

def main():
    # Example URL
    url = "https://www.99acres.com/4-bhk-bedroom-independent-house-villa-for-sale-in-heritage-villa-76-kasindra-ahmedabad-south-3798-sq-ft-npspid-D85437812"
    
    # 1. Fetch
    # Note: If running locally without internet, ensure 'sample_property_page.html' exists
    try:
        with open("sample_property_page.html", "r", encoding="utf-8") as f:
            html = f.read()
        print("✅ Loaded local HTML file")
    except FileNotFoundError:
        print("⚠️ Local file not found, fetching from URL...")
        html = call_API(url)

    if not html:
        return

    # 2. Extract & Parse
    try:
        initial_data = extract_initial_data(html)
        payload = build_payload(initial_data)

        print("\n🎯 FINAL EXTRACTED DATA:\n")
        print(json.dumps(payload, indent=2, ensure_ascii=False))
        
    except RuntimeError as e:
        print(e)
    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")

if __name__ == "__main__":
   main()

✅ Loaded local HTML file
❌ window.__initialData__ not found in page HTML


In [1]:
import requests

url = "https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-dev-the-galaxy-shela-ahmedabad-west-2010-sq-ft-npspid-Q86774166"

payload = {}
headers = {
  'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
  'accept-language': 'en-US,en;q=0.9',
  'cache-control': 'max-age=0',
  'dnt': '1',
  'if-none-match': 'W/"a6005-ZjAUJ6UqcAVkoD/Ia9vK1KAzt6Y"',
  'priority': 'u=0, i',
  'sec-ch-ua': '"Not(A:Brand";v="8", "Chromium";v="144", "Google Chrome";v="144"',
  'sec-ch-ua-mobile': '?0',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-dest': 'document',
  'sec-fetch-mode': 'navigate',
  'sec-fetch-site': 'none',
  'sec-fetch-user': '?1',
  'upgrade-insecure-requests': '1',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36',
  'Cookie': '99_ab=29; GOOGLE_SEARCH_ID=4084631769856564402; xAB=SuperControlGroup%3D17%3AN%2CtopMatchHandlingAB%3D66%3AD%2CBFdataremoval%3D29%3AY%2CseamlessLogin%3D56%3AY%2CEMAILOPTIONAL%3D48%3AY%2CMLSEARCHSRP%3D1%3AY%2CDSSimilarProperties%3D87%3AY%2CshowInhousePlayer%3D38%3AD%2CIATABSVF%3D36%3AD%2CVSRAlgoDemandShaping%3D88%3AY%2CBUILDERFLOORSRP%3D99%3AN%2CMLSEARCHMONET%3D78%3AY%2CNEARBYSRP%3D31%3AY%2CppfTemplatePostingV2%3D48%3AY%2CbrokerSupplyRef%3D40%3AY%2CownerEmailOptional%3D3%3AY%2CppfCommSoftPosting%3D51%3AN; session_source=DIRECT; landmark_toast=true; _gcl_au=1.1.349653589.1769856570; _gid=GA1.2.649331308.1769856571; _fbp=fb.1.1769856570831.879743020712544429; _clck=1yly54u%5E2%5Eg36%5E0%5E2222; showCookieBanner=1; _hjSessionUser_3171461=eyJpZCI6IjlmNzI5MTZkLTUwMTMtNTVjNC1iMmQzLTZhMTkxOGYzNGFhYiIsImNyZWF0ZWQiOjE3Njk4NTY1NzEwMTksImV4aXN0aW5nIjp0cnVlfQ==; 99_ab=29; acceptedMobileDsiclaimer=true; hp_bcf_data=; _hjSession_3171461=eyJpZCI6IjBmZWM5ZDM4LWEwZTktNGY3My1iZWYyLWJhZjgyYWI2ZjFiOCIsImMiOjE3Njk4OTMyODc3NjcsInMiOjAsInIiOjAsInNiIjowLCJzciI6MCwic2UiOjAsImZzIjowfQ==; CPN=/4-bhk-bedroom-independent-house-villa-for-sale-in-heritage-villa-76-kasindra-ahmedabad-south-3798-sq-ft-npspid-S87884256; sessionno=11; session30m=eyJ0b2tlbklkIjoiMWZiZjEyOGYtNTU2Yy00YmNiLTlhNjAtNjQ4MDM2NmEyYmU0IiwiaXNzdWVEYXRlIjoxNzY5ODk1NTM3NDM3fQ; _sess_id=QJp7rZLVDuTmZxk1xcmJuON5cuVMNchpKs7BM6POpQ5eImBNoJipYqYneLZmueoXApnP7MTfnT7Vh11iOYkvaQ%3D%3D; _ga=GA1.1.477790739.1769856571; _ga_9QHC0XEKPS=GS2.1.s1769895558$o6$g1$t1769896413$j40$l0$h0; _uetsid=85d4df90fe9211f0b03a6d756b7dcf35; _uetvid=85d55230fe9211f08d2da945ba1deab4; _clsk=90czva%5E1769896415033%5E7%5E0%5Es.clarity.ms%2Fcollect; 99_ab=74; GOOGLE_SEARCH_ID=2104631769791185608; sessionno=3; xAB=SuperControlGroup%3D17%3AN%2CtopMatchHandlingAB%3D66%3AD%2CBFdataremoval%3D29%3AY%2CseamlessLogin%3D56%3AY%2CEMAILOPTIONAL%3D48%3AY%2CMLSEARCHSRP%3D1%3AY%2CDSSimilarProperties%3D87%3AY%2CshowInhousePlayer%3D38%3AD%2CIATABSVF%3D36%3AD%2CVSRAlgoDemandShaping%3D88%3AY%2CBUILDERFLOORSRP%3D99%3AN%2CMLSEARCHMONET%3D78%3AY%2CNEARBYSRP%3D31%3AY%2CppfTemplatePostingV2%3D48%3AY%2CbrokerSupplyRef%3D40%3AY%2CownerEmailOptional%3D3%3AY%2CppfCommSoftPosting%3D51%3AN'
}

response = requests.request("GET", url, headers=headers, data=payload)

# print(response.text)


In [3]:
data_list =  ['412724', 'F87169838', 'muteUnmuteButton_1', 'J87356456', 'muteUnmuteButton_2', 'r2mWidget_245177', 'r2mWidget_388209', 'r2mWidget_395198', 'r2mWidget_389184', 'r2mWidget_398882', 'r2mWidget_3330', 'r2mWidget_393086', 'r2mWidget_371407', 'r2mWidget_450209', 'r2mWidget_396286', 'r2mWidget_384222', 'r2mWidget_388125', '441674', 'S87884256', 'muteUnmuteButton_4', 'SRP_REI_WIDGET', 'SRP_REI_LOCALITIES_CARD', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITIES_CARD', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITIES_CARD', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'SRP_REI_LOCALITY', 'SRP_REI_VIEW_PROPERTIES', 'C87304328', 'muteUnmuteButton_5', 'O85189794', 'muteUnmuteButton_6', 'K85243782', '422985', 'A86715928', 'muteUnmuteButton_9', '414571', 'B87453554', 'muteUnmuteButton_11', 'R88029114', 'I87260730', '438445', '458948', '411626', 'npSrpWidget', 'crossSellWidget_399222', 'crossSellWidget_412524', 'crossSellWidget_426492', 'crossSellWidget_435798', 'crossSellWidget_417593', 'crossSellWidget_441347', 'crossSellWidget_444754', 'crossSellWidget_430758', 'crossSellWidget_415878', 'crossSellWidget_376686', 'crossSellWidget_376214', 'crossSellWidget_423203', 'crossSellWidget_432390', 'crossSellWidget_389205', 'crossSellWidget_442250', 'crossSellWidget_373269', 'crossSellWidget_398157', 'crossSellWidget_398013', 'crossSellWidget_329399', 'crossSellWidget_423087', '451210', 'M86635630', 'muteUnmuteButton_18', '398004', 'M87409938', 'muteUnmuteButton_20', '408242', 'C83000646', 'muteUnmuteButton_22', '426600', '410305']

In [ ]:
def clean_and_validate_widgets(data_list):
    exclude_patterns = ['muteUnmuteButton_', 'crossSellWidget_' , 'r2mWidget_']
    """
    1. Removes items containing specific keywords.
    2. Removes items that do not contain at least one numeric digit.
    """
    return [
        item for item in data_list 
        if not any(pattern in item for pattern in exclude_patterns) # Filter keywords
        and any(char.isdigit() for char in item)                    # Must have a digit
    ]

# Implementation

final_list = clean_and_validate_widgets(data_list)
print(final_list)

['412724', 'F87169838', 'J87356456', '441674', 'S87884256', 'C87304328', 'O85189794', 'K85243782', '422985', 'A86715928', '414571', 'B87453554', 'R88029114', 'I87260730', '438445', '458948', '411626', '451210', 'M86635630', '398004', 'M87409938', '408242', 'C83000646', '426600', '410305']
